# Bayesian SSE Association Models

In [1]:
from __future__ import annotations

from collections.abc import Iterable, Sequence
from pathlib import Path
import re
import sys

import arviz as az
import bambi as bmb
from IPython.display import display
import numpy as np
import pandas as pd
from scipy.special import logit as logit_func

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "utils").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "utils").exists():
    raise RuntimeError("Run this notebook from inside the scotland repository.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sse_detection.lib import (  # noqa: E402
    DEFAULT_MIXING_FEATURES,
    add_observed_mixing_entropy_scales,
    HIGH_PRIORITY_CANDIDATE_TIERS,
    load_sse_outputs,
    load_sequence_data,
)

SSE_OUTPUT_DIR = PROJECT_ROOT / "sse_detection" / "results" / "sse_outputs"

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [2]:
STANDARDISE_SPECS = {
    "z_wn_prop_sequenced": "wn_prop_sequenced",
    "z_log1p_wn_positive_tests": "log1p_wn_positive_tests",
    "z_dz_cum_prop_sequenced": "dz_cum_prop_sequenced",
    "z_dz_cum_incidence_per_capita": "dz_cum_incidence_per_capita",
    "z_dz_7d_test_positivity": "dz_7d_test_positivity",
    "z_log1p_dz_cum_positive_tests": "log1p_dz_cum_positive_tests",
    "z_dz_cum_prop_vaccinated": "dz_cum_prop_vaccinated",
}

COMPOSITION_SPECS = [
    {
        "name": "sex",
        "column": "sex",
        "reference": "Male",
        "label": "Sex",
    },
    {
        "name": "age_band",
        "column": "age_band",
        "reference": "20-24",
        "label": "Age band",
    },
    {
        "name": "simd_quintile",
        "column": "dz_simd_quintile",
        "reference": 1,
        "label": "SIMD quintile",
    },
    {
        "name": "urban_rural_class",
        "column": "dz_urban_rural_class",
        "reference": "Large Urban Areas",
        "label": "Urban/rural class",
    },
    {
        "name": "health_board",
        "column": "dz_health_board",
        "reference": "Greater Glasgow and Clyde",
        "label": "Health board",
    },
]

EXPANDED_CONTEXT_ADJUSTERS = [
    "z_dz_cum_prop_sequenced",
    "z_dz_cum_incidence_per_capita",
    "z_dz_7d_test_positivity",
    "z_log1p_dz_cum_positive_tests",
]


def add_standardised_adjusters(data: pd.DataFrame) -> pd.DataFrame:
    """Add standardised surveillance and context adjusters used by notebooks."""
    out = add_observed_mixing_entropy_scales(data)
    if "wn_positive_tests" in out.columns:
        out["log1p_wn_positive_tests"] = np.log1p(out["wn_positive_tests"])
    if "dz_cum_positive_tests" in out.columns:
        out["log1p_dz_cum_positive_tests"] = np.log1p(out["dz_cum_positive_tests"])
    for target, source in STANDARDISE_SPECS.items():
        if source not in out.columns:
            continue
        values = out[source].astype(float)
        sd = values.std(skipna=True)
        if pd.isna(sd) or sd == 0:
            out[target] = np.nan
        else:
            out[target] = (values - values.mean(skipna=True)) / sd
    return out

In [3]:
RANDOM_SEED = 123
CLUSTER_ID_COL = "cluster_id"
MIXING_GROUP_VARS = ("window_idx", "clade")
COMP_GROUP_VARS = MIXING_GROUP_VARS + (CLUSTER_ID_COL,)

RESULT_DIR = PROJECT_ROOT / "sse_detection" / "results" / "bayesian_socio_geo_demo"

COMPOSITION_PREDICTORS = {
    spec["column"]: spec["reference"] for spec in COMPOSITION_SPECS
}
MIXING_PREDICTORS = list(DEFAULT_MIXING_FEATURES)
EXPANDED_ADJUSTERS = list(EXPANDED_CONTEXT_ADJUSTERS)

# Narrow these lists for a smoke-test grid, e.g. ["sex"] or ["sex_entropy_z"].
COMPOSITION_GRID_PREDICTORS = list(COMPOSITION_PREDICTORS)
MIXING_GRID_PREDICTORS = list(MIXING_PREDICTORS)

# Test-run controls. When USE_DEV_SAMPLE=True, model fitting uses balanced-ish
# samples created by sample_model_test_data; the full complete-case frames are
# still retained for reporting and for later full-data runs.
USE_DEV_SAMPLE = True
DEV_SAMPLE_ROWS = 1_000
COMPOSITION_DEV_SAMPLE_ROWS = DEV_SAMPLE_ROWS * 2
MIXING_DEV_SAMPLE_ROWS = DEV_SAMPLE_ROWS
DEV_SAMPLE_POSITIVE_FRACTION = 0.25

RUN_COMPOSITION_GRID = True
RUN_MIXING_GRID = True

SAVE_INFERENCE_DATA = False
RETRY_SINGLE_CORE_ON_EOF = True

FULL_SAMPLING_KWARGS = {
    "draws": 2_000,
    "tune": 2_000,
    "chains": 4,
    "cores": 4,
    "target_accept": 0.99,
    "random_seed": RANDOM_SEED,
}

DEV_SAMPLING_KWARGS = {
    "draws": 500,
    "tune": 500,
    "chains": 2,
    "cores": 1,
    "target_accept": 0.99,
    "random_seed": RANDOM_SEED,
}

SAMPLING_KWARGS = DEV_SAMPLING_KWARGS if USE_DEV_SAMPLE else FULL_SAMPLING_KWARGS

## Load and Align Data

The node outcome follows the main pipeline: high-priority burst or burden candidates among nodes at least as large as the smallest high-priority candidate. Sequence-level models inherit the candidate label from their cluster.


In [4]:
sse_outputs = load_sse_outputs(SSE_OUTPUT_DIR)
cluster_data = add_standardised_adjusters(sse_outputs.cluster_table.copy())
sequence_data = add_standardised_adjusters(load_sequence_data())

cluster_data["candidate"] = cluster_data["candidate_tier"].isin(
    HIGH_PRIORITY_CANDIDATE_TIERS
)

candidate_sizes = cluster_data.loc[cluster_data["candidate"], "cluster_size"].dropna()
if candidate_sizes.empty:
    raise ValueError("No high-priority candidate nodes were found.")

min_candidate_size = int(candidate_sizes.min())
eligible_nodes = cluster_data.loc[
    cluster_data["cluster_size"].ge(min_candidate_size)
].copy()

eligible_sequence_data = sequence_data.merge(
    eligible_nodes[[CLUSTER_ID_COL, "candidate"]],
    on=CLUSTER_ID_COL,
    how="inner",
)

candidate_node_rate = float(eligible_nodes["candidate"].mean())
candidate_sequence_rate = float(eligible_sequence_data["candidate"].mean())

candidate_summary = pd.DataFrame(
    [
        {
            "dataset": "eligible_nodes",
            "rows": len(eligible_nodes),
            "candidate_rate": candidate_node_rate,
            "candidates": int(eligible_nodes["candidate"].sum()),
        },
        {
            "dataset": "eligible_sequence_data",
            "rows": len(eligible_sequence_data),
            "candidate_rate": candidate_sequence_rate,
            "candidates": int(eligible_sequence_data["candidate"].sum()),
        },
    ]
)
display(candidate_summary)

,dataset,rows,candidate_rate,candidates
0,eligible_nodes,13059,0.047017,614
1,eligible_sequence_data,264139,0.253049,66840


## Data and Formula Helpers


In [5]:
def _unique_preserve_order(items: Iterable[str]) -> list[str]:
    """Return unique strings in first-seen order."""
    out: list[str] = []
    seen: set[str] = set()
    for item in items:
        if item not in seen:
            out.append(item)
            seen.add(item)
    return out


def _categories_from(series: pd.Series) -> list:
    """Use declared categories when present; otherwise preserve observed order."""
    if isinstance(series.dtype, pd.CategoricalDtype):
        return list(series.cat.categories)
    return series.dropna().drop_duplicates().tolist()


def sample_model_test_data(
    data: pd.DataFrame,
    *,
    outcome: str = "candidate",
    max_rows: int = DEV_SAMPLE_ROWS,
    positive_fraction: float = 0.35,
    random_state: int = RANDOM_SEED,
    categorical_vars: Sequence[str] | None = None,
) -> pd.DataFrame:
    """Return a balanced-ish development sample covering observed categories."""
    positives = data.loc[data[outcome] == 1]
    negatives = data.loc[data[outcome] == 0]

    if positives.empty or negatives.empty:
        raise ValueError(f"'{outcome}' must contain at least one 0 and one 1.")

    positive_fraction = float(np.clip(positive_fraction, 0.01, 0.99))
    n_pos = min(len(positives), max(1, int(round(max_rows * positive_fraction))))
    n_neg = min(len(negatives), max_rows - n_pos)
    if n_neg <= 0:
        raise ValueError("Sampling settings left no room for negative controls.")

    category_cols = [
        col
        for col in _unique_preserve_order(categorical_vars or ())
        if col in data.columns
    ]
    category_levels = {
        col: [
            level
            for level in _categories_from(data[col])
            if data[col].eq(level).any()
        ]
        for col in category_cols
    }

    missing_by_col = {col: set(levels) for col, levels in category_levels.items()}
    seed_indices: list[object] = []
    selected_indices: set[object] = set()

    for col, levels in category_levels.items():
        for level in levels:
            if level not in missing_by_col[col]:
                continue

            candidates = data.loc[data[col].eq(level)]
            coverage = pd.Series(0, index=candidates.index, dtype=int)
            for cover_col, missing_levels in missing_by_col.items():
                if missing_levels:
                    coverage += candidates[cover_col].isin(missing_levels).astype(int)

            best_indices = coverage.loc[coverage.eq(coverage.max())].index
            seed_row = data.loc[best_indices].sample(
                n=1,
                random_state=random_state + len(seed_indices),
                replace=False,
            )
            idx = seed_row.index[0]
            if idx not in selected_indices:
                seed_indices.append(idx)
                selected_indices.add(idx)

            row = data.loc[idx]
            for cover_col in category_cols:
                missing_by_col[cover_col].discard(row[cover_col])

    if len(seed_indices) > max_rows:
        raise ValueError(
            "max_rows is too small to include all observed categorical levels "
            f"({len(seed_indices)} seed rows needed; max_rows={max_rows})."
        )

    sampled_parts = [data.loc[seed_indices]] if seed_indices else []
    seed_outcomes = (
        data.loc[seed_indices, outcome]
        if seed_indices
        else pd.Series(dtype=int)
    )
    seed_pos = int((seed_outcomes == 1).sum())
    seed_neg = int((seed_outcomes == 0).sum())
    remaining_capacity = max_rows - len(seed_indices)

    n_pos_fill = max(0, n_pos - seed_pos)
    n_neg_fill = max(0, n_neg - seed_neg)
    if n_pos_fill + n_neg_fill > remaining_capacity:
        if seed_pos >= n_pos:
            n_pos_fill = 0
            n_neg_fill = remaining_capacity
        elif seed_neg >= n_neg:
            n_neg_fill = 0
            n_pos_fill = remaining_capacity
        else:
            scale = remaining_capacity / (n_pos_fill + n_neg_fill)
            n_pos_fill = int(round(n_pos_fill * scale))
            n_neg_fill = remaining_capacity - n_pos_fill

    def sample_available(frame: pd.DataFrame, n: int, seed_offset: int) -> pd.DataFrame:
        available = frame.drop(index=list(selected_indices), errors="ignore")
        n = min(n, len(available))
        if n <= 0:
            return available.iloc[0:0]
        out = available.sample(
            n=n,
            random_state=random_state + seed_offset,
            replace=False,
        )
        selected_indices.update(out.index.tolist())
        return out

    sampled_parts.append(sample_available(positives, n_pos_fill, 10_000))
    sampled_parts.append(sample_available(negatives, n_neg_fill, 20_000))

    sampled = pd.concat(sampled_parts, axis=0)
    if len(sampled) < max_rows:
        top_up = sample_available(data, max_rows - len(sampled), 30_000)
        sampled = pd.concat([sampled, top_up], axis=0)

    sampled = sampled.sample(frac=1, random_state=random_state).reset_index(drop=True)

    for col in category_cols:
        sampled[col] = pd.Categorical(
            sampled[col],
            categories=_categories_from(data[col]),
        )

    return sampled


def get_complete_case_data(
    df: pd.DataFrame,
    *,
    outcome: str = "candidate",
    predictors: Iterable[str] | None = None,
    group_vars: Sequence[str] = MIXING_GROUP_VARS,
    categorical_vars: Sequence[str] = (),
    id_cols: Sequence[str] = (CLUSTER_ID_COL,),
    verbose: bool = True,
) -> pd.DataFrame:
    """Create a complete-case model frame with stable categorical dtypes."""
    if predictors is None:
        raise ValueError("Please provide predictor columns.")

    required_cols = _unique_preserve_order(
        [outcome, *id_cols, *list(predictors), *group_vars]
    )
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in dataframe: {missing_cols}")

    before_n = len(df)
    out = df.loc[:, required_cols].dropna().copy()
    after_n = len(out)

    out[outcome] = out[outcome].astype(int)

    categorical_cols = _unique_preserve_order(
        [*categorical_vars, *group_vars, *id_cols]
    )
    for col in categorical_cols:
        if col in out.columns:
            out[col] = pd.Categorical(out[col], categories=_categories_from(df[col]))

    if verbose:
        print("Complete-case summary")
        print("---------------------")
        print(f"Rows before dropna: {before_n:,}")
        print(f"Rows after dropna:  {after_n:,}")
        print(f"Rows dropped:       {before_n - after_n:,}")
        print(f"Percent retained:   {after_n / before_n:.1%}")
        print(f"Candidate rate:     {out[outcome].mean():.2%}")
        print("\nGrouping levels:")
        for col in group_vars:
            print(f"{col}: {out[col].nunique()} levels")

    return out


def treatment_term(variable: str, reference: str | int | float | None) -> str:
    """Return a Bambi/formulae categorical term with an optional reference level."""
    if reference is None:
        return f"C({variable})"
    return f"C({variable}, Treatment(reference={reference!r}))"


def formula_with_varying_intercepts(
    outcome: str,
    terms: str | Sequence[str],
    *,
    group_vars: Sequence[str] = MIXING_GROUP_VARS,
) -> str:
    """Build a logistic mixed-model formula with window/clade intercepts."""
    if isinstance(terms, str):
        terms = [terms]
    rhs_terms = [*terms, *(f"(1|{group})" for group in group_vars)]
    return f"{outcome} ~ " + " + ".join(rhs_terms)


def compact_categorical_levels(
    data: pd.DataFrame,
    columns: Sequence[str],
) -> pd.DataFrame:
    """Drop unobserved categories for sampled grouping variables."""
    out = data.copy()
    for col in columns:
        if col not in out.columns:
            continue
        if isinstance(out[col].dtype, pd.CategoricalDtype):
            out[col] = out[col].cat.remove_unused_categories()
        else:
            out[col] = pd.Categorical(out[col])
    return out


def make_model_fit_data(
    data: pd.DataFrame,
    *,
    label: str,
    use_dev_sample: bool,
    max_rows: int,
    positive_fraction: float = DEV_SAMPLE_POSITIVE_FRACTION,
    categorical_vars: Sequence[str] = (),
    compact_category_vars: Sequence[str] = (),
    random_state: int = RANDOM_SEED,
) -> pd.DataFrame:
    """Return either the full model frame or a sampled frame for smoke tests."""
    if not use_dev_sample:
        print(f"{label}: using full frame ({len(data):,} rows).")
        return data.copy()

    sampled = sample_model_test_data(
        data,
        max_rows=max_rows,
        positive_fraction=positive_fraction,
        random_state=random_state,
        categorical_vars=categorical_vars,
    )
    sampled = compact_categorical_levels(sampled, compact_category_vars)
    print(
        f"{label}: using development sample "
        f"({len(sampled):,} of {len(data):,} rows; "
        f"candidate rate {sampled['candidate'].mean():.2%})."
    )
    return sampled


def model_frame_summary_row(
    label: str,
    full_df: pd.DataFrame,
    fit_df: pd.DataFrame,
    *,
    outcome: str = "candidate",
) -> dict[str, object]:
    """Summarise the full and fitting frames side by side."""
    return {
        "frame": label,
        "full_rows": len(full_df),
        "fit_rows": len(fit_df),
        "fit_fraction": len(fit_df) / len(full_df) if len(full_df) else np.nan,
        "full_candidates": int(full_df[outcome].sum()),
        "fit_candidates": int(fit_df[outcome].sum()),
        "full_candidate_rate": float(full_df[outcome].mean()),
        "fit_candidate_rate": float(fit_df[outcome].mean()),
        "use_dev_sample": len(fit_df) != len(full_df),
    }


## Bambi Fitting and Summaries


In [6]:
_RANDOM_EFFECT_RE = re.compile(r"\([^|()]+\|([^()]+)\)")


def random_effect_groups(formula: str) -> list[str]:
    """Extract grouping variables from terms such as ``(1|window_idx)``."""
    return [match.strip() for match in _RANDOM_EFFECT_RE.findall(formula)]


def fit_bayesian_logistic_model(
    data: pd.DataFrame,
    formula: str,
    *,
    family: str = "bernoulli",
    categorical: Sequence[str] | None = None,
    fixed_prior_sigma: float = 1.0,
    intercept_prior_sigma: float = 1.5,
    random_effect_sigma: float = 1.0,
    draws: int = 2_000,
    tune: int = 2_000,
    chains: int = 4,
    cores: int = 4,
    target_accept: float = 0.99,
    random_seed: int = RANDOM_SEED,
    log_likelihood: bool = True,
    noncentered: bool = True,
    retry_single_core_on_eof: bool = RETRY_SINGLE_CORE_ON_EOF,
) -> tuple[bmb.Model, az.InferenceData]:
    """Fit a Bambi hierarchical logistic regression with weakly informative priors."""
    if "~" not in formula:
        raise ValueError("Formula must contain '~', e.g. 'candidate ~ x + (1|group)'.")

    response = formula.split("~", 1)[0].strip()
    if response not in data.columns:
        raise ValueError(f"Response variable '{response}' not found in data.")

    outcome_mean = float(np.clip(data[response].mean(), 1e-6, 1 - 1e-6))
    priors = {
        "Intercept": bmb.Prior(
            "Normal",
            mu=logit_func(outcome_mean),
            sigma=intercept_prior_sigma,
        ),
        "common": bmb.Prior("Normal", mu=0, sigma=fixed_prior_sigma),
        "group_specific": bmb.Prior(
            "Normal",
            mu=0,
            sigma=bmb.Prior("HalfNormal", sigma=random_effect_sigma),
        ),
    }

    group_cols = random_effect_groups(formula)
    categorical_cols = _unique_preserve_order([*(categorical or ()), *group_cols])

    model = bmb.Model(
        formula=formula,
        data=data,
        family=family,
        priors=priors,
        categorical=categorical_cols or None,
        noncentered=noncentered,
    )
    fit_kwargs = {
        "draws": draws,
        "tune": tune,
        "chains": chains,
        "cores": cores,
        "target_accept": target_accept,
        "random_seed": random_seed,
        "idata_kwargs": {"log_likelihood": log_likelihood},
    }
    try:
        idata = model.fit(**fit_kwargs)
    except EOFError:
        if not retry_single_core_on_eof or cores <= 1:
            raise
        print(
            "Parallel PyMC sampling ended with EOFError; "
            "retrying the same model with cores=1."
        )
        fit_kwargs["cores"] = 1
        idata = model.fit(**fit_kwargs)
    return model, idata


def _print_section(title: str, char: str = "=") -> None:
    print(f"\n{title}")
    print(char * len(title))


def _format_df_for_print(
    df: pd.DataFrame,
    *,
    float_digits: int = 4,
    width: int = 160,
    max_colwidth: int = 80,
) -> str:
    with pd.option_context(
        "display.max_rows",
        None,
        "display.max_columns",
        None,
        "display.width",
        width,
        "display.max_colwidth",
        max_colwidth,
        "display.float_format",
        lambda x: f"{x:,.{float_digits}f}",
    ):
        return df.to_string()


def _show_table(
    df: pd.DataFrame,
    *,
    display_tables: bool = False,
    float_digits: int = 4,
) -> None:
    if display_tables:
        display(df.style.format(precision=float_digits))
    else:
        print(_format_df_for_print(df, float_digits=float_digits))


def _available_posterior_vars(
    idata: az.InferenceData,
    var_names: Sequence[str] | None,
) -> list[str] | None:
    if var_names is None:
        return None
    available = set(idata.posterior.data_vars)  # type: ignore
    selected = [var for var in var_names if var in available]
    missing = [var for var in var_names if var not in available]
    if missing:
        print("Skipping unavailable posterior variables:")
        for var in missing:
            print(f"  - {var}")
    if not selected:
        raise KeyError("None of the requested posterior variables were found.")
    return selected


def summarise_bambi_idata(
    idata: az.InferenceData,
    *,
    var_names: Sequence[str] | None = None,
    hdi_prob: float = 0.95,
    odds_ratio_vars: Sequence[str] | None = None,
    print_diagnostics: bool = True,
    rhat_threshold: float = 1.01,
    ess_threshold: int = 400,
    display_tables: bool = False,
    float_digits: int = 4,
) -> pd.DataFrame:
    """Print diagnostics and return a focused ArviZ posterior summary."""
    selected_vars = _available_posterior_vars(idata, var_names)
    summary = az.summary(
        idata,
        var_names=selected_vars,
        hdi_prob=hdi_prob,
        round_to=float_digits,
    )
    diagnostic_summary = az.summary(
        idata,
        hdi_prob=hdi_prob,
        round_to=float_digits,
    )

    if print_diagnostics:
        _print_section("Bayesian model diagnostics")
        diagnostic_rows = []

        if hasattr(idata, "sample_stats") and "diverging" in idata.sample_stats:  # type: ignore
            n_div = int(idata.sample_stats["diverging"].sum().item())  # type: ignore
            n_total = int(idata.sample_stats["diverging"].size)  # type: ignore
            div_rate = n_div / n_total
            diagnostic_rows.append(
                {
                    "Diagnostic": "Divergences",
                    "Value": f"{n_div} / {n_total} ({div_rate:.2%})",
                    "Status": "OK" if n_div == 0 else "WARNING",
                    "Interpretation": (
                        "No divergent transitions."
                        if n_div == 0
                        else "Investigate divergent transitions."
                    ),
                }
            )

        try:
            bfmi = np.asarray(az.bfmi(idata))
            min_bfmi = float(np.nanmin(bfmi))
            bfmi_by_chain = ", ".join(f"{x:.3f}" for x in bfmi)
            diagnostic_rows.append(
                {
                    "Diagnostic": "BFMI",
                    "Value": f"min={min_bfmi:.3f}; chains=[{bfmi_by_chain}]",
                    "Status": "OK" if min_bfmi >= 0.3 else "WARNING",
                    "Interpretation": (
                        "Energy exploration looks acceptable."
                        if min_bfmi >= 0.3
                        else "One or more chains have BFMI < 0.3."
                    ),
                }
            )
        except Exception as exc:
            diagnostic_rows.append(
                {
                    "Diagnostic": "BFMI",
                    "Value": "Could not compute",
                    "Status": "NA",
                    "Interpretation": str(exc),
                }
            )

        if "r_hat" in diagnostic_summary.columns:
            max_rhat = float(diagnostic_summary["r_hat"].max(skipna=True))
            diagnostic_rows.append(
                {
                    "Diagnostic": "Max R-hat",
                    "Value": f"{max_rhat:.4f}",
                    "Status": "OK" if max_rhat <= rhat_threshold else "WARNING",
                    "Interpretation": (
                        f"All posterior variables are at or below {rhat_threshold}."
                        if max_rhat <= rhat_threshold
                        else f"Some posterior variables exceed {rhat_threshold}."
                    ),
                }
            )

        if "ess_bulk" in diagnostic_summary.columns:
            min_bulk_ess = float(diagnostic_summary["ess_bulk"].min(skipna=True))
            diagnostic_rows.append(
                {
                    "Diagnostic": "Min bulk ESS",
                    "Value": f"{min_bulk_ess:.1f}",
                    "Status": "OK" if min_bulk_ess >= ess_threshold else "WARNING",
                    "Interpretation": (
                        f"All bulk ESS values are at least {ess_threshold}."
                        if min_bulk_ess >= ess_threshold
                        else f"Some bulk ESS values are below {ess_threshold}."
                    ),
                }
            )

        if "ess_tail" in diagnostic_summary.columns:
            min_tail_ess = float(diagnostic_summary["ess_tail"].min(skipna=True))
            diagnostic_rows.append(
                {
                    "Diagnostic": "Min tail ESS",
                    "Value": f"{min_tail_ess:.1f}",
                    "Status": "OK" if min_tail_ess >= ess_threshold else "WARNING",
                    "Interpretation": (
                        f"All tail ESS values are at least {ess_threshold}."
                        if min_tail_ess >= ess_threshold
                        else f"Some tail ESS values are below {ess_threshold}."
                    ),
                }
            )

        if hasattr(idata, "sample_stats") and "tree_depth" in idata.sample_stats:  # type: ignore
            max_tree_depth = int(idata.sample_stats["tree_depth"].max().item())  # type: ignore
            diagnostic_rows.append(
                {
                    "Diagnostic": "Max tree depth",
                    "Value": str(max_tree_depth),
                    "Status": "INFO",
                    "Interpretation": "Maximum observed tree depth.",
                }
            )

        _show_table(
            pd.DataFrame(diagnostic_rows),
            display_tables=display_tables,
            float_digits=float_digits,
        )

        _print_section("Posterior summary")
        _show_table(
            summary,  # type: ignore
            display_tables=display_tables,
            float_digits=float_digits,
        )

    if odds_ratio_vars is not None:
        _print_section("Odds-ratio summaries")
        available = set(idata.posterior.data_vars)  # type: ignore
        for var in odds_ratio_vars:
            if var not in available:
                print(f"\n{var}: not found in idata.posterior")
                continue

            beta = idata.posterior[var]  # type: ignore
            odds_ratio = np.exp(beta)
            or_idata = odds_ratio.to_dataset(name=f"OR_{var}")
            or_summary = az.summary(
                or_idata,
                hdi_prob=hdi_prob,
                round_to=float_digits,
            )

            prob_df = pd.DataFrame(
                {
                    "Quantity": [
                        "P(beta > 0 | data)",
                        "P(beta < 0 | data)",
                        "P(OR > 1 | data)",
                        "P(OR < 1 | data)",
                    ],
                    "Probability": [
                        float((beta > 0).mean().item()),
                        float((beta < 0).mean().item()),
                        float((odds_ratio > 1).mean().item()),
                        float((odds_ratio < 1).mean().item()),
                    ],
                }
            )

            _print_section(var, char="-")
            _show_table(
                or_summary,  # type: ignore
                display_tables=display_tables,
                float_digits=float_digits,
            )
            _show_table(
                prob_df,
                display_tables=display_tables,
                float_digits=float_digits,
            )

    return summary  # type: ignore


def fit_and_summarise_model(
    data: pd.DataFrame,
    formula: str,
    *,
    var_names: Sequence[str],
    odds_ratio_vars: Sequence[str],
    categorical: Sequence[str] = MIXING_GROUP_VARS,
    display_tables: bool = True,
    **fit_kwargs,
) -> dict[str, object]:
    """Fit a model and keep the model, posterior, formula, and focused summary."""
    model, idata = fit_bayesian_logistic_model(
        data=data,
        formula=formula,
        categorical=categorical,
        **fit_kwargs,
    )
    summary = summarise_bambi_idata(
        idata,
        var_names=var_names,
        hdi_prob=0.95,
        odds_ratio_vars=odds_ratio_vars,
        display_tables=display_tables,
    )
    response = formula.split("~", 1)[0].strip()
    return {
        "formula": formula,
        "model": model,
        "idata": idata,
        "summary": summary,
        "var_names": list(var_names),
        "odds_ratio_vars": list(odds_ratio_vars),
        "n_rows": len(data),
        "candidate_rate": float(data[response].mean()),
    }


## Composition Models: Sequence-Level Association

Composition models ask whether individual sequence attributes are associated with membership in a candidate cluster, with varying intercepts for window and clade.


In [7]:
unknown_composition_predictors = sorted(
    set(COMPOSITION_GRID_PREDICTORS) - set(COMPOSITION_PREDICTORS)
)
if unknown_composition_predictors:
    raise ValueError(
        f"Unknown COMPOSITION_GRID_PREDICTORS: {unknown_composition_predictors}"
    )

composition_terms = {
    column: treatment_term(column, COMPOSITION_PREDICTORS[column])
    for column in COMPOSITION_GRID_PREDICTORS
}
composition_categorical_vars = (*COMP_GROUP_VARS, *COMPOSITION_GRID_PREDICTORS)

composition_primary_df = get_complete_case_data(
    df=eligible_sequence_data,
    outcome="candidate",
    predictors=COMPOSITION_GRID_PREDICTORS,
    group_vars=COMP_GROUP_VARS,
    categorical_vars=composition_categorical_vars,
)

composition_expanded_df = get_complete_case_data(
    df=eligible_sequence_data,
    outcome="candidate",
    predictors=[*COMPOSITION_GRID_PREDICTORS, *EXPANDED_ADJUSTERS],
    group_vars=COMP_GROUP_VARS,
    categorical_vars=composition_categorical_vars,
)

composition_primary_fit_df = make_model_fit_data(
    composition_primary_df,
    label="composition_primary",
    use_dev_sample=USE_DEV_SAMPLE,
    max_rows=COMPOSITION_DEV_SAMPLE_ROWS,
    positive_fraction=DEV_SAMPLE_POSITIVE_FRACTION,
    categorical_vars=COMPOSITION_GRID_PREDICTORS,
    compact_category_vars=COMP_GROUP_VARS,
)
composition_expanded_fit_df = make_model_fit_data(
    composition_expanded_df,
    label="composition_expanded",
    use_dev_sample=USE_DEV_SAMPLE,
    max_rows=COMPOSITION_DEV_SAMPLE_ROWS,
    positive_fraction=DEV_SAMPLE_POSITIVE_FRACTION,
    categorical_vars=COMPOSITION_GRID_PREDICTORS,
    compact_category_vars=COMP_GROUP_VARS,
)

composition_fit_frame_summary = pd.DataFrame(
    [
        model_frame_summary_row(
            "composition_primary", composition_primary_df, composition_primary_fit_df
        ),
        model_frame_summary_row(
            "composition_expanded", composition_expanded_df, composition_expanded_fit_df
        ),
    ]
)
display(composition_fit_frame_summary)

composition_single_primary_formulas = {
    column: formula_with_varying_intercepts(
        "candidate", term, group_vars=COMP_GROUP_VARS
    )
    for column, term in composition_terms.items()
}
composition_single_expanded_formulas = {
    column: formula_with_varying_intercepts(
        "candidate",
        [term, *EXPANDED_ADJUSTERS],
        group_vars=COMP_GROUP_VARS,
    )
    for column, term in composition_terms.items()
}
joint_composition_primary_formula = formula_with_varying_intercepts(
    "candidate",
    list(composition_terms.values()),
    group_vars=COMP_GROUP_VARS,
)
joint_composition_expanded_formula = formula_with_varying_intercepts(
    "candidate",
    [*composition_terms.values(), *EXPANDED_ADJUSTERS],
    group_vars=COMP_GROUP_VARS,
)

composition_model_df = composition_primary_fit_df
composition_formulas = composition_single_primary_formulas
joint_composition_formula = joint_composition_primary_formula

composition_model_grid = pd.DataFrame(
    [
        {"model": "single_primary", "predictor": column, "formula": formula}
        for column, formula in composition_single_primary_formulas.items()
    ]
    + [
        {"model": "single_expanded", "predictor": column, "formula": formula}
        for column, formula in composition_single_expanded_formulas.items()
    ]
    + [
        {
            "model": "joint_primary",
            "predictor": "all_composition_predictors",
            "formula": joint_composition_primary_formula,
        },
        {
            "model": "joint_expanded",
            "predictor": "all_composition_predictors",
            "formula": joint_composition_expanded_formula,
        },
    ]
)
display(composition_model_grid)


Complete-case summary
---------------------
Rows before dropna: 264,139
Rows after dropna:  264,139
Rows dropped:       0
Percent retained:   100.0%
Candidate rate:     25.30%

Grouping levels:
window_idx: 67 levels
clade: 21 levels
cluster_id: 13059 levels
Complete-case summary
---------------------
Rows before dropna: 264,139
Rows after dropna:  264,119
Rows dropped:       20
Percent retained:   100.0%
Candidate rate:     25.31%

Grouping levels:
window_idx: 67 levels
clade: 21 levels
cluster_id: 13059 levels
composition_primary: using development sample (2,000 of 264,139 rows; candidate rate 25.00%).
composition_expanded: using development sample (2,000 of 264,119 rows; candidate rate 25.00%).


,frame,full_rows,fit_rows,fit_fraction,full_candidates,fit_candidates,full_candidate_rate,fit_candidate_rate,use_dev_sample
0,composition_primary,264139,2000,0.007572,66840,500,0.253049,0.25,True
1,composition_expanded,264119,2000,0.007572,66840,500,0.253068,0.25,True


,model,predictor,formula
0,single_primary,sex,"candidate ~ C(sex, Treatment(reference='Male')..."
1,single_primary,age_band,"candidate ~ C(age_band, Treatment(reference='2..."
2,single_primary,dz_simd_quintile,"candidate ~ C(dz_simd_quintile, Treatment(refe..."
3,single_primary,dz_urban_rural_class,"candidate ~ C(dz_urban_rural_class, Treatment(..."
4,single_primary,dz_health_board,"candidate ~ C(dz_health_board, Treatment(refer..."
5,single_expanded,sex,"candidate ~ C(sex, Treatment(reference='Male')..."
6,single_expanded,age_band,"candidate ~ C(age_band, Treatment(reference='2..."
7,single_expanded,dz_simd_quintile,"candidate ~ C(dz_simd_quintile, Treatment(refe..."
8,single_expanded,dz_urban_rural_class,"candidate ~ C(dz_urban_rural_class, Treatment(..."
9,single_expanded,dz_health_board,"candidate ~ C(dz_health_board, Treatment(refer..."


In [8]:
composition_single_primary_results: dict[str, dict[str, object]] = {}
composition_single_expanded_results: dict[str, dict[str, object]] = {}

if RUN_COMPOSITION_GRID:
    for col, formula in composition_single_primary_formulas.items():
        term = composition_terms[col]
        print(f"\nFitting primary single-predictor composition model: {col}")
        composition_single_primary_results[col] = fit_and_summarise_model(
            data=composition_primary_fit_df,
            formula=formula,
            var_names=[
                "Intercept",
                term,
                "1|window_idx_sigma",
                "1|clade_sigma",
                "1|cluster_id_sigma",
            ],
            odds_ratio_vars=[term],
            categorical=COMP_GROUP_VARS,
            **SAMPLING_KWARGS,
        )

    for col, formula in composition_single_expanded_formulas.items():
        term = composition_terms[col]
        print(f"\nFitting expanded single-predictor composition model: {col}")
        composition_single_expanded_results[col] = fit_and_summarise_model(
            data=composition_expanded_fit_df,
            formula=formula,
            var_names=[
                "Intercept",
                term,
                *EXPANDED_ADJUSTERS,
                "1|window_idx_sigma",
                "1|clade_sigma",
                "1|cluster_id_sigma",
            ],
            odds_ratio_vars=[term],
            categorical=COMP_GROUP_VARS,
            **SAMPLING_KWARGS,
        )
else:
    print("RUN_COMPOSITION_GRID=False; skipping single-predictor composition sampling.")

composition_single_results = composition_single_primary_results


Modeling the probability that candidate==1



Fitting primary single-predictor composition model: sex


Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [Intercept, C(sex, Treatment(reference='Male')), 1|window_idx_sigma, 1|window_idx_offset, 1|clade_sigma, 1|clade_offset, 1|cluster_id_sigma, 1|cluster_id_offset]


Output()

Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 44 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


...


Posterior summary


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,-6.4682,0.6083,-7.6084,-5.3173,0.0213,0.0166,824.6270,771.1038,1.0020
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Ayrshire and Arran]",0.0355,0.5989,-1.1354,1.1246,0.0165,0.0165,1315.3568,664.0742,0.9993
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Borders]",-0.2627,0.9475,-2.2677,1.4665,0.0222,0.0378,1782.1089,832.5885,1.0012
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Dumfries and Galloway]",-0.1008,0.7707,-1.5092,1.5332,0.0191,0.0194,1639.5741,707.0696,1.0005
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Fife]",-0.6467,0.6581,-2.0117,0.5028,0.0162,0.0246,1630.1070,679.0110,1.0065
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Forth Valley]",0.3703,0.6403,-0.8853,1.5945,0.0179,0.0182,1268.7459,799.8616,1.0005
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Grampian]",0.1780,0.6103,-1.0725,1.2979,0.0181,0.0154,1157.3290,799.1280,0.9990
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Highland]",0.2670,0.6962,-0.9906,1.6411,0.0198,0.0185,1221.0448,840.4975,1.0005
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Lanarkshire]",0.1276,0.5208,-0.8961,1.1827,0.0153,0.0218,1183.5401,558.0821,1.0045
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Lothian]",-0.2602,0.4657,-1.1226,0.6891,0.0126,0.0132,1365.5607,709.4283,1.0026



Odds-ratio summaries

C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))
--------------------------------------------------------------------


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Ayrshire and Arran]",1.2338,0.7657,0.1295,2.7328,0.0211,0.0293,1315.3568,664.0742,0.9992
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Borders]",1.2212,1.6339,0.0546,3.8018,0.0555,0.2296,1782.1089,832.5885,1.0021
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Dumfries and Galloway]",1.2162,1.0705,0.1178,3.1709,0.0287,0.0634,1639.5741,707.0696,1.0011
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Fife]",0.6505,0.4980,0.0767,1.5064,0.0180,0.0608,1630.1070,679.0110,1.0107
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Forth Valley]",1.7948,1.4893,0.2527,4.1216,0.0426,0.1955,1268.7459,799.8616,1.0005
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Grampian]",1.4323,0.9116,0.2015,3.2149,0.0269,0.0399,1157.3290,799.1280,0.9997
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Highland]",1.6540,1.2278,0.1922,3.9816,0.0361,0.0616,1221.0448,840.4975,1.0005
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Lanarkshire]",1.3037,0.7519,0.2468,2.7016,0.0316,0.0616,1183.5401,558.0821,1.0055
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Lothian]",0.8595,0.4223,0.2773,1.8046,0.0125,0.0180,1365.5607,709.4283,1.0032
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Orkney]",1.4382,1.4774,0.0601,4.3532,0.0453,0.0998,1726.8911,702.3021,1.0016


,Quantity,Probability
0,P(beta > 0 | data),0.5139
1,P(beta < 0 | data),0.4861
2,P(OR > 1 | data),0.5139
3,P(OR < 1 | data),0.4861


In [9]:
composition_joint_primary_results: dict[str, dict[str, object]] = {}
composition_joint_expanded_results: dict[str, dict[str, object]] = {}

if RUN_COMPOSITION_GRID:
    print("\nFitting joint primary composition model")
    composition_joint_primary_results["joint_primary"] = fit_and_summarise_model(
        data=composition_primary_fit_df,
        formula=joint_composition_primary_formula,
        var_names=[
            "Intercept",
            *composition_terms.values(),
            "1|window_idx_sigma",
            "1|clade_sigma",
            "1|cluster_id_sigma",
        ],
        odds_ratio_vars=list(composition_terms.values()),
        categorical=COMP_GROUP_VARS,
        **SAMPLING_KWARGS,
    )

    print("\nFitting joint expanded composition model")
    composition_joint_expanded_results["joint_expanded"] = fit_and_summarise_model(
        data=composition_expanded_fit_df,
        formula=joint_composition_expanded_formula,
        var_names=[
            "Intercept",
            *composition_terms.values(),
            *EXPANDED_ADJUSTERS,
            "1|window_idx_sigma",
            "1|clade_sigma",
            "1|cluster_id_sigma",
        ],
        odds_ratio_vars=list(composition_terms.values()),
        categorical=COMP_GROUP_VARS,
        **SAMPLING_KWARGS,
    )
else:
    print("RUN_COMPOSITION_GRID=False; skipping joint composition-model sampling.")

joint_composition_results = composition_joint_primary_results


Modeling the probability that candidate==1



Fitting joint primary composition model


Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [Intercept, C(sex, Treatment(reference='Male')), C(age_band, Treatment(reference='20-24')), C(dz_simd_quintile, Treatment(reference=1)), C(dz_urban_rural_class, Treatment(reference='Large Urban Areas')), C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde')), 1|window_idx_sigma, 1|window_idx_offset, 1|clade_sigma, 1|clade_offset, 1|cluster_id_sigma, 1|cluster_id_offset]


Output()

Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 67 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


...

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
"OR_C(dz_urban_rural_class, Treatment(reference='Large Urban Areas'))[Accessible Rural]",0.8662,0.5421,0.1550,1.9761,0.0161,0.0270,1019.2888,903.9393,0.9990
"OR_C(dz_urban_rural_class, Treatment(reference='Large Urban Areas'))[Accessible Small Towns]",0.9877,0.6772,0.1768,2.3678,0.0198,0.0361,1204.5970,764.5027,0.9995
"OR_C(dz_urban_rural_class, Treatment(reference='Large Urban Areas'))[Other Urban Areas]",1.3908,0.7051,0.3508,2.6809,0.0245,0.0409,898.3168,618.0046,1.0008
"OR_C(dz_urban_rural_class, Treatment(reference='Large Urban Areas'))[Remote Rural]",1.4814,1.2135,0.0925,4.0593,0.0391,0.0699,1423.3096,667.2253,1.0029
"OR_C(dz_urban_rural_class, Treatment(reference='Large Urban Areas'))[Remote Small Towns]",1.3309,1.1813,0.1134,3.4982,0.0370,0.0777,1200.9050,770.0919,1.0011


,Quantity,Probability
0,P(beta > 0 | data),0.4820
1,P(beta < 0 | data),0.5180
2,P(OR > 1 | data),0.4820
3,P(OR < 1 | data),0.5180



C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))
--------------------------------------------------------------------


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Ayrshire and Arran]",1.2432,1.0855,0.1646,2.7318,0.0416,0.1947,1043.0495,756.3399,1.0014
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Borders]",1.2007,1.3069,0.0777,3.8110,0.0370,0.0778,2346.5720,944.9944,0.9985
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Dumfries and Galloway]",1.2194,1.1666,0.0973,3.5004,0.0343,0.0826,1964.7954,801.2942,0.9987
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Fife]",0.6462,0.4844,0.0816,1.5829,0.0139,0.0249,1463.2622,850.5549,1.0064
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Forth Valley]",1.8445,1.5387,0.2065,4.6798,0.0526,0.1261,1124.4292,790.4124,1.0012
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Grampian]",1.5554,1.1944,0.1916,3.7726,0.0367,0.0872,1319.8740,736.3441,0.9999
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Highland]",1.7032,1.4429,0.1637,4.6248,0.0445,0.0938,1437.3419,792.4024,1.0011
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Lanarkshire]",1.2360,0.7622,0.2329,2.8378,0.0236,0.0330,977.2917,789.3797,1.0018
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Lothian]",0.9477,0.5307,0.2531,2.0371,0.0163,0.0255,1315.9980,762.1006,1.0062
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Orkney]",1.4626,1.7196,0.0579,4.4018,0.0506,0.1461,2619.4940,762.1057,0.9998


,Quantity,Probability
0,P(beta > 0 | data),0.5037
1,P(beta < 0 | data),0.4963
2,P(OR > 1 | data),0.5037
3,P(OR < 1 | data),0.4963


## Mixing Models: Node-Level Association

Mixing models ask whether candidate nodes have unusual cluster-level entropy or context profiles, again with varying intercepts for window and clade.


In [10]:
unknown_mixing_predictors = sorted(set(MIXING_GRID_PREDICTORS) - set(MIXING_PREDICTORS))
if unknown_mixing_predictors:
    raise ValueError(f"Unknown MIXING_GRID_PREDICTORS: {unknown_mixing_predictors}")

mixing_primary_df = get_complete_case_data(
    df=eligible_nodes,
    outcome="candidate",
    predictors=MIXING_GRID_PREDICTORS,
    group_vars=MIXING_GROUP_VARS,
    categorical_vars=MIXING_GROUP_VARS,
)

mixing_expanded_df = get_complete_case_data(
    df=eligible_nodes,
    outcome="candidate",
    predictors=[*MIXING_GRID_PREDICTORS, *EXPANDED_ADJUSTERS],
    group_vars=MIXING_GROUP_VARS,
    categorical_vars=MIXING_GROUP_VARS,
)

mixing_primary_fit_df = make_model_fit_data(
    mixing_primary_df,
    label="mixing_primary",
    use_dev_sample=USE_DEV_SAMPLE,
    max_rows=MIXING_DEV_SAMPLE_ROWS,
    positive_fraction=DEV_SAMPLE_POSITIVE_FRACTION,
    compact_category_vars=MIXING_GROUP_VARS,
)
mixing_expanded_fit_df = make_model_fit_data(
    mixing_expanded_df,
    label="mixing_expanded",
    use_dev_sample=USE_DEV_SAMPLE,
    max_rows=MIXING_DEV_SAMPLE_ROWS,
    positive_fraction=DEV_SAMPLE_POSITIVE_FRACTION,
    compact_category_vars=MIXING_GROUP_VARS,
)

mixing_fit_frame_summary = pd.DataFrame(
    [
        model_frame_summary_row(
            "mixing_primary", mixing_primary_df, mixing_primary_fit_df
        ),
        model_frame_summary_row(
            "mixing_expanded", mixing_expanded_df, mixing_expanded_fit_df
        ),
    ]
)
display(mixing_fit_frame_summary)

mixing_single_primary_formulas = {
    col: formula_with_varying_intercepts("candidate", col)
    for col in MIXING_GRID_PREDICTORS
}
mixing_single_expanded_formulas = {
    col: formula_with_varying_intercepts(
        "candidate",
        [col, *EXPANDED_ADJUSTERS],
    )
    for col in MIXING_GRID_PREDICTORS
}
joint_mixing_primary_formula = formula_with_varying_intercepts(
    "candidate",
    MIXING_GRID_PREDICTORS,
)
joint_mixing_expanded_formula = formula_with_varying_intercepts(
    "candidate",
    [*MIXING_GRID_PREDICTORS, *EXPANDED_ADJUSTERS],
)

mixing_model_grid = pd.DataFrame(
    [
        {"model": "single_primary", "predictor": col, "formula": formula}
        for col, formula in mixing_single_primary_formulas.items()
    ]
    + [
        {"model": "single_expanded", "predictor": col, "formula": formula}
        for col, formula in mixing_single_expanded_formulas.items()
    ]
    + [
        {
            "model": "joint_primary",
            "predictor": "all_mixing_predictors",
            "formula": joint_mixing_primary_formula,
        },
        {
            "model": "joint_expanded",
            "predictor": "all_mixing_predictors",
            "formula": joint_mixing_expanded_formula,
        },
    ]
)
display(mixing_model_grid)


Complete-case summary
---------------------
Rows before dropna: 13,059
Rows after dropna:  12,525
Rows dropped:       534
Percent retained:   95.9%
Candidate rate:     4.82%

Grouping levels:
window_idx: 58 levels
clade: 19 levels
Complete-case summary
---------------------
Rows before dropna: 13,059
Rows after dropna:  12,525
Rows dropped:       534
Percent retained:   95.9%
Candidate rate:     4.82%

Grouping levels:
window_idx: 58 levels
clade: 19 levels
mixing_primary: using development sample (1,000 of 12,525 rows; candidate rate 25.00%).
mixing_expanded: using development sample (1,000 of 12,525 rows; candidate rate 25.00%).


,frame,full_rows,fit_rows,fit_fraction,full_candidates,fit_candidates,full_candidate_rate,fit_candidate_rate,use_dev_sample
0,mixing_primary,12525,1000,0.07984,604,250,0.048224,0.25,True
1,mixing_expanded,12525,1000,0.07984,604,250,0.048224,0.25,True


,model,predictor,formula
0,single_primary,sex_entropy_z,candidate ~ sex_entropy_z + (1|window_idx) + (...
1,single_primary,age_entropy_z,candidate ~ age_entropy_z + (1|window_idx) + (...
2,single_primary,simd_entropy_z,candidate ~ simd_entropy_z + (1|window_idx) + ...
3,single_primary,datazone_entropy_z,candidate ~ datazone_entropy_z + (1|window_idx...
4,single_primary,local_authority_entropy_z,candidate ~ local_authority_entropy_z + (1|win...
5,single_primary,urban_rural_entropy_z,candidate ~ urban_rural_entropy_z + (1|window_...
6,single_primary,health_board_entropy_z,candidate ~ health_board_entropy_z + (1|window...
7,single_primary,vaccination_entropy_z,candidate ~ vaccination_entropy_z + (1|window_...
8,single_expanded,sex_entropy_z,candidate ~ sex_entropy_z + z_dz_cum_prop_sequ...
9,single_expanded,age_entropy_z,candidate ~ age_entropy_z + z_dz_cum_prop_sequ...


In [11]:
single_primary_results: dict[str, dict[str, object]] = {}

if RUN_MIXING_GRID:
    for col, formula in mixing_single_primary_formulas.items():
        print(f"\nFitting primary mixing model: {col}")
        single_primary_results[col] = fit_and_summarise_model(
            data=mixing_primary_fit_df,
            formula=formula,
            var_names=[
                "Intercept",
                col,
                "1|window_idx_sigma",
                "1|clade_sigma",
            ],
            odds_ratio_vars=[col],
            categorical=MIXING_GROUP_VARS,
            **SAMPLING_KWARGS,
        )
else:
    print("RUN_MIXING_GRID=False; skipping primary single-predictor sampling.")


Modeling the probability that candidate==1



Fitting primary mixing model: sex_entropy_z


Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [Intercept, sex_entropy_z, 1|window_idx_sigma, 1|window_idx_offset, 1|clade_sigma, 1|clade_offset]


Output()

Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 12 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


...


Posterior summary


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,-1.1207,0.1037,-1.3554,-0.9240,0.0048,0.0042,502.7208,379.3917,1.0011
vaccination_entropy_z,0.0012,0.0623,-0.1215,0.1172,0.0018,0.0018,1143.1698,790.7773,1.0040
1|window_idx_sigma,0.0983,0.0727,0.0005,0.2448,0.0030,0.0022,436.2881,415.2203,1.0060
1|clade_sigma,0.1314,0.1273,0.0000,0.3936,0.0068,0.0068,306.5831,347.3109,1.0020



Odds-ratio summaries

vaccination_entropy_z
---------------------


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
OR_vaccination_entropy_z,1.0032,0.0626,0.8856,1.1244,0.0019,0.0019,1143.1698,790.7773,1.0040


,Quantity,Probability
0,P(beta > 0 | data),0.5050
1,P(beta < 0 | data),0.4950
2,P(OR > 1 | data),0.5050
3,P(OR < 1 | data),0.4950


In [12]:
single_expanded_results: dict[str, dict[str, object]] = {}

if RUN_MIXING_GRID:
    for col, formula in mixing_single_expanded_formulas.items():
        print(f"\nFitting expanded mixing model: {col}")
        single_expanded_results[col] = fit_and_summarise_model(
            data=mixing_expanded_fit_df,
            formula=formula,
            var_names=[
                "Intercept",
                col,
                *EXPANDED_ADJUSTERS,
                "1|window_idx_sigma",
                "1|clade_sigma",
            ],
            odds_ratio_vars=[col],
            categorical=MIXING_GROUP_VARS,
            **SAMPLING_KWARGS,
        )
else:
    print("RUN_MIXING_GRID=False; skipping expanded single-predictor sampling.")


Modeling the probability that candidate==1



Fitting expanded mixing model: sex_entropy_z


Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [Intercept, sex_entropy_z, z_dz_cum_prop_sequenced, z_dz_cum_incidence_per_capita, z_dz_7d_test_positivity, z_log1p_dz_cum_positive_tests, 1|window_idx_sigma, 1|window_idx_offset, 1|clade_sigma, 1|clade_offset]


Output()

Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 18 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


...


Posterior summary


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,-1.1315,0.1166,-1.3388,-0.8631,0.0043,0.0064,974.5759,656.0483,1.0002
vaccination_entropy_z,0.0019,0.0590,-0.1191,0.1113,0.0012,0.0021,2571.8291,635.6650,1.0048
z_dz_cum_prop_sequenced,0.2338,0.1772,-0.1110,0.5750,0.0042,0.0057,1793.6632,836.7620,1.0012
z_dz_cum_incidence_per_capita,-0.1436,0.2774,-0.6267,0.4319,0.0085,0.0104,1084.3382,651.7386,1.0011
z_dz_7d_test_positivity,-0.0706,0.1593,-0.3515,0.2526,0.0035,0.0050,2068.7030,814.7415,1.0061
z_log1p_dz_cum_positive_tests,0.3007,0.2653,-0.1725,0.8424,0.0081,0.0092,1072.6065,609.5765,1.0021
1|window_idx_sigma,0.1038,0.0745,0.0003,0.2510,0.0026,0.0018,697.3515,590.5536,1.0022
1|clade_sigma,0.1540,0.1578,0.0009,0.4706,0.0066,0.0095,525.7688,559.6430,1.0041



Odds-ratio summaries

vaccination_entropy_z
---------------------


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
OR_vaccination_entropy_z,1.0036,0.0593,0.8877,1.1177,0.0012,0.0021,2571.8291,635.6650,1.0049


,Quantity,Probability
0,P(beta > 0 | data),0.5060
1,P(beta < 0 | data),0.4940
2,P(OR > 1 | data),0.5060
3,P(OR < 1 | data),0.4940


In [13]:
joint_primary_results: dict[str, dict[str, object]] = {}
joint_expanded_results: dict[str, dict[str, object]] = {}

if RUN_MIXING_GRID:
    print("\nFitting joint primary mixing model")
    joint_primary_results["joint_primary"] = fit_and_summarise_model(
        data=mixing_primary_fit_df,
        formula=joint_mixing_primary_formula,
        var_names=[
            "Intercept",
            *MIXING_GRID_PREDICTORS,
            "1|window_idx_sigma",
            "1|clade_sigma",
        ],
        odds_ratio_vars=MIXING_GRID_PREDICTORS,
        categorical=MIXING_GROUP_VARS,
        **SAMPLING_KWARGS,
    )

    print("\nFitting joint expanded mixing model")
    joint_expanded_results["joint_expanded"] = fit_and_summarise_model(
        data=mixing_expanded_fit_df,
        formula=joint_mixing_expanded_formula,
        var_names=[
            "Intercept",
            *MIXING_GRID_PREDICTORS,
            *EXPANDED_ADJUSTERS,
            "1|window_idx_sigma",
            "1|clade_sigma",
        ],
        odds_ratio_vars=MIXING_GRID_PREDICTORS,
        categorical=MIXING_GROUP_VARS,
        **SAMPLING_KWARGS,
    )
else:
    print("RUN_MIXING_GRID=False; skipping joint mixing-model sampling.")


Modeling the probability that candidate==1



Fitting joint primary mixing model


Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [Intercept, sex_entropy_z, age_entropy_z, simd_entropy_z, datazone_entropy_z, local_authority_entropy_z, urban_rural_entropy_z, health_board_entropy_z, vaccination_entropy_z, 1|window_idx_sigma, 1|window_idx_offset, 1|clade_sigma, 1|clade_offset]


Output()

Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 34 seconds.
There were 889 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


...

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
OR_health_board_entropy_z,1.6577,2.1633,0.0371,5.1539,0.1318,0.3156,292.8422,296.6811,1.0156


,Quantity,Probability
0,P(beta > 0 | data),0.4860
1,P(beta < 0 | data),0.5140
2,P(OR > 1 | data),0.4860
3,P(OR < 1 | data),0.5140



vaccination_entropy_z
---------------------


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
OR_vaccination_entropy_z,1.7500,2.0750,0.0374,5.0915,0.1092,0.1960,230.0251,279.9326,1.0153


,Quantity,Probability
0,P(beta > 0 | data),0.5200
1,P(beta < 0 | data),0.4800
2,P(OR > 1 | data),0.5200
3,P(OR < 1 | data),0.4800


## Save Bayesian Model Results

The cell below writes model grids, fitting-frame summaries, posterior summaries, odds-ratio summaries, sampler diagnostics, and optional ArviZ `InferenceData` files to `sse_detection/results/bayesian_socio_geo_demo`.


In [14]:
def safe_result_name(value: object) -> str:
    """Return a filesystem-safe name for a model or predictor label."""
    cleaned = re.sub(r"[^A-Za-z0-9_.=-]+", "_", str(value)).strip("_")
    return cleaned or "model"


def posterior_odds_ratio_summary(
    idata: az.InferenceData,
    var_names: Sequence[str],
    *,
    hdi_prob: float = 0.95,
) -> pd.DataFrame:
    """Summarise exponentiated fixed effects and posterior direction probabilities."""
    rows: list[dict[str, object]] = []
    available = set(idata.posterior.data_vars)  # type: ignore

    for var in var_names:
        if var not in available:
            continue

        beta = idata.posterior[var]  # type: ignore
        coefficient_dims = [dim for dim in beta.dims if dim not in {"chain", "draw"}]
        if coefficient_dims:
            beta_iter = beta.stack(__coefficient__=coefficient_dims)
            selectors = range(beta_iter.sizes["__coefficient__"])
        else:
            beta_iter = beta
            selectors = [None]

        for selector in selectors:
            if selector is None:
                beta_component = beta_iter
                parameter = var
            else:
                beta_component = beta_iter.isel(__coefficient__=selector)
                coord = beta_component["__coefficient__"].item()
                if isinstance(coord, tuple):
                    suffix = ", ".join(str(item) for item in coord)
                else:
                    suffix = str(coord)
                parameter = f"{var}[{suffix}]"

            beta_samples = np.asarray(beta_component).reshape(-1)
            odds_ratio_samples = np.exp(beta_samples)
            hdi = az.hdi(odds_ratio_samples, hdi_prob=hdi_prob)
            rows.append(
                {
                    "term": var,
                    "parameter": parameter,
                    "mean": float(np.mean(odds_ratio_samples)),
                    "sd": float(np.std(odds_ratio_samples, ddof=1)),
                    f"hdi_{(1 - hdi_prob) / 2:.1%}": float(hdi[0]),
                    f"hdi_{1 - (1 - hdi_prob) / 2:.1%}": float(hdi[1]),
                    "p_beta_gt_0": float(np.mean(beta_samples > 0)),
                    "p_beta_lt_0": float(np.mean(beta_samples < 0)),
                    "p_or_gt_1": float(np.mean(odds_ratio_samples > 1)),
                    "p_or_lt_1": float(np.mean(odds_ratio_samples < 1)),
                    "n_samples": int(odds_ratio_samples.size),
                }
            )

    return pd.DataFrame(rows)


def sampler_diagnostics_summary(
    idata: az.InferenceData,
    *,
    rhat_threshold: float = 1.01,
    ess_threshold: int = 400,
) -> pd.DataFrame:
    """Create the same core sampler diagnostics used in printed model summaries."""
    rows: list[dict[str, object]] = []
    diagnostic_summary = az.summary(idata, round_to=4)

    if hasattr(idata, "sample_stats") and "diverging" in idata.sample_stats:  # type: ignore
        n_div = int(idata.sample_stats["diverging"].sum().item())  # type: ignore
        n_total = int(idata.sample_stats["diverging"].size)  # type: ignore
        rows.append(
            {
                "diagnostic": "divergences",
                "value": n_div,
                "total": n_total,
                "status": "OK" if n_div == 0 else "WARNING",
            }
        )

    try:
        bfmi = np.asarray(az.bfmi(idata))
        rows.append(
            {
                "diagnostic": "bfmi_min",
                "value": float(np.nanmin(bfmi)),
                "total": np.nan,
                "status": "OK" if float(np.nanmin(bfmi)) >= 0.3 else "WARNING",
            }
        )
    except Exception as exc:
        rows.append(
            {
                "diagnostic": "bfmi_min",
                "value": np.nan,
                "total": np.nan,
                "status": f"NA: {exc}",
            }
        )

    if "r_hat" in diagnostic_summary.columns:
        max_rhat = float(diagnostic_summary["r_hat"].max(skipna=True))
        rows.append(
            {
                "diagnostic": "max_rhat",
                "value": max_rhat,
                "total": np.nan,
                "status": "OK" if max_rhat <= rhat_threshold else "WARNING",
            }
        )

    if "ess_bulk" in diagnostic_summary.columns:
        min_bulk_ess = float(diagnostic_summary["ess_bulk"].min(skipna=True))
        rows.append(
            {
                "diagnostic": "min_bulk_ess",
                "value": min_bulk_ess,
                "total": np.nan,
                "status": "OK" if min_bulk_ess >= ess_threshold else "WARNING",
            }
        )

    if "ess_tail" in diagnostic_summary.columns:
        min_tail_ess = float(diagnostic_summary["ess_tail"].min(skipna=True))
        rows.append(
            {
                "diagnostic": "min_tail_ess",
                "value": min_tail_ess,
                "total": np.nan,
                "status": "OK" if min_tail_ess >= ess_threshold else "WARNING",
            }
        )

    return pd.DataFrame(rows)


def write_summary_table(summary: pd.DataFrame, path: Path) -> None:
    """Write an ArviZ summary while preserving parameter names from the index."""
    out = summary.copy()
    out.insert(0, "parameter", out.index.astype(str))
    out.to_csv(path, index=False)


def save_model_result(
    result: dict[str, object],
    *,
    domain: str,
    model_set: str,
    predictor: str,
    output_dir: Path = RESULT_DIR,
    save_idata: bool = SAVE_INFERENCE_DATA,
) -> dict[str, object]:
    """Save one fitted model result and return a manifest row."""
    model_dir = output_dir / domain / model_set / safe_result_name(predictor)
    model_dir.mkdir(parents=True, exist_ok=True)

    summary = result.get("summary")
    if isinstance(summary, pd.DataFrame):
        write_summary_table(summary, model_dir / "posterior_summary.csv")

    idata = result.get("idata")
    odds_ratio_vars = result.get("odds_ratio_vars", [])
    if isinstance(idata, az.InferenceData):
        odds_ratio_summary = posterior_odds_ratio_summary(idata, odds_ratio_vars)  # type: ignore[arg-type]
        if not odds_ratio_summary.empty:
            odds_ratio_summary.to_csv(model_dir / "odds_ratio_summary.csv", index=False)

        diagnostics = sampler_diagnostics_summary(idata)
        diagnostics.to_csv(model_dir / "sampler_diagnostics.csv", index=False)

        if save_idata:
            idata.to_netcdf(model_dir / "idata.nc")  # type: ignore

    metadata = pd.DataFrame(
        [
            {
                "domain": domain,
                "model_set": model_set,
                "predictor": predictor,
                "formula": result.get("formula"),
                "n_rows": result.get("n_rows"),
                "candidate_rate": result.get("candidate_rate"),
                "use_dev_sample": USE_DEV_SAMPLE,
            }
        ]
    )
    metadata.to_csv(model_dir / "metadata.csv", index=False)

    return {
        "domain": domain,
        "model_set": model_set,
        "predictor": predictor,
        "model_dir": str(model_dir.relative_to(PROJECT_ROOT)),
        "n_rows": result.get("n_rows"),
        "candidate_rate": result.get("candidate_rate"),
    }


RESULT_DIR.mkdir(parents=True, exist_ok=True)

composition_model_grid.to_csv(RESULT_DIR / "composition_model_grid.csv", index=False)
mixing_model_grid.to_csv(RESULT_DIR / "mixing_model_grid.csv", index=False)

model_frame_summary = pd.concat(
    [composition_fit_frame_summary, mixing_fit_frame_summary],
    ignore_index=True,
)
model_frame_summary.to_csv(RESULT_DIR / "model_frame_summary.csv", index=False)

run_config = pd.DataFrame(
    [
        {
            "use_dev_sample": USE_DEV_SAMPLE,
            "composition_dev_sample_rows": COMPOSITION_DEV_SAMPLE_ROWS,
            "mixing_dev_sample_rows": MIXING_DEV_SAMPLE_ROWS,
            "dev_sample_positive_fraction": DEV_SAMPLE_POSITIVE_FRACTION,
            "run_composition_grid": RUN_COMPOSITION_GRID,
            "run_mixing_grid": RUN_MIXING_GRID,
            "composition_grid_predictors": ",".join(COMPOSITION_GRID_PREDICTORS),
            "mixing_grid_predictors": ",".join(MIXING_GRID_PREDICTORS),
            "sampling_kwargs": repr(SAMPLING_KWARGS),
            "save_inference_data": SAVE_INFERENCE_DATA,
        }
    ]
)
run_config.to_csv(RESULT_DIR / "run_config.csv", index=False)

result_groups = [
    ("composition", "single_primary", composition_single_primary_results),
    ("composition", "single_expanded", composition_single_expanded_results),
    ("composition", "joint_primary", composition_joint_primary_results),
    ("composition", "joint_expanded", composition_joint_expanded_results),
    ("mixing", "single_primary", single_primary_results),
    ("mixing", "single_expanded", single_expanded_results),
    ("mixing", "joint_primary", joint_primary_results),
    ("mixing", "joint_expanded", joint_expanded_results),
]

manifest_rows = []
for domain, model_set, results in result_groups:
    for predictor, result in results.items():
        manifest_rows.append(
            save_model_result(
                result,
                domain=domain,
                model_set=model_set,
                predictor=predictor,
            )
        )

saved_model_manifest = pd.DataFrame(
    manifest_rows,
    columns=[
        "domain",
        "model_set",
        "predictor",
        "model_dir",
        "n_rows",
        "candidate_rate",
    ],
)
saved_model_manifest.to_csv(RESULT_DIR / "saved_model_manifest.csv", index=False)

print(
    f"Saved Bayesian socio-geo-demo outputs to: {RESULT_DIR.relative_to(PROJECT_ROOT)}"
)
display(saved_model_manifest)

Saved Bayesian socio-geo-demo outputs to: sse_detection/results/bayesian_socio_geo_demo


,domain,model_set,predictor,model_dir,n_rows,candidate_rate
0,composition,single_primary,sex,sse_detection/results/bayesian_socio_geo_demo/...,2000,0.25
1,composition,single_primary,age_band,sse_detection/results/bayesian_socio_geo_demo/...,2000,0.25
2,composition,single_primary,dz_simd_quintile,sse_detection/results/bayesian_socio_geo_demo/...,2000,0.25
3,composition,single_primary,dz_urban_rural_class,sse_detection/results/bayesian_socio_geo_demo/...,2000,0.25
4,composition,single_primary,dz_health_board,sse_detection/results/bayesian_socio_geo_demo/...,2000,0.25
5,composition,single_expanded,sex,sse_detection/results/bayesian_socio_geo_demo/...,2000,0.25
6,composition,single_expanded,age_band,sse_detection/results/bayesian_socio_geo_demo/...,2000,0.25
7,composition,single_expanded,dz_simd_quintile,sse_detection/results/bayesian_socio_geo_demo/...,2000,0.25
8,composition,single_expanded,dz_urban_rural_class,sse_detection/results/bayesian_socio_geo_demo/...,2000,0.25
9,composition,single_expanded,dz_health_board,sse_detection/results/bayesian_socio_geo_demo/...,2000,0.25
